In [1]:
%reload_ext dotenv
%dotenv

import warnings
import logging
import datetime
import os

from faster_whisper import WhisperModel
from utils.utils import mp42wav, cut_blanks, audio_paths

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
warnings.filterwarnings(action="ignore")   # <--- ignore after imports

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
warnings.filterwarnings(action="ignore")   # <--- ignore after imports

cannot find .env file


In [2]:
ASR_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"
MIN_SILENCE_LEN = 500
SILENCE_THRESH = -60
# ASR_MODEL = "XA9/Belle-faster-whisper-large-v3-zh-punct"

WHISPER_MODEL = WhisperModel(ASR_MODEL, device="cuda", compute_type="float16")

In [3]:
def mfa(audio_file_dir):
    os.system(
        f"mfa align --output_format json \
                --use_threading \
                --use_mp \
                --overwrite \
                --clean \
                --final_clean \
                {audio_file_dir}/chunks \
                mandarin_china_mfa \
                mandarin_mfa \
                {audio_file_dir}/chunks"
    )

In [4]:
def transcribe_audio(audio_file, subtitle_format="srt"):
    segments, info = WHISPER_MODEL.transcribe(
        f"{audio_file}.wav",
        word_timestamps=True,
        initial_prompt="以下是普通话的句子。",
        beam_size=5,
        language="zh",
        max_new_tokens=433,
        condition_on_previous_text=False,
        vad_filter=False,
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    sub_list: list[dict[str, str]] = []
    srt_content = ""
    srt_number = 0
    for segment in segments:
        start_time_str = format_to_srt(segment.start)
        end_time_str = format_to_srt(segment.end)
        sub_text = replace_special_chars(segment.text)
        print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, sub_text))
        sub_entry = {
            "start_time_str": start_time_str,
            "end_time_str": end_time_str,
            "text": sub_text,
        }
        sub_list.append(sub_entry)  # Add formatted subtitles to list

    if subtitle_format == "srt":
        for sub in sub_list:  # Add subtitle's index number
            sub_srt = f"{sub['start_time_str']} --> {sub['end_time_str']}\n{sub['text']}\n\n"
            srt_content += str(srt_number) + "\n" + sub_srt
            srt_number = srt_number + 1
        with open(f"{audio_file}.srt", "w", encoding="utf-8") as srt_file:
            srt_file.write(srt_content)
    elif subtitle_format == "json":
        with open(f"{audio_file}.json", "w", encoding="utf-8") as json_file:
            json_file.write(str(sub_list).replace("'", '"'))
    elif subtitle_format == "txt":
        with open(f"{audio_file}.txt", "w", encoding="utf-8") as txt_file:
            for sub in sub_list:
                txt_file.write(sub["text"])
    print("")
    print("Saved: " + os.path.abspath(f"{audio_file}.{subtitle_format}"))


def replace_special_chars(
    text,
):  # remove space and "! " if the first letter is space or "! "
    # Check if the text starts with "!" or " " and ends with " "
    if text.startswith("! ") or text.startswith(" "):
        # Replace the special characters with an empty string
        text = text.replace("!", "").replace(
            " ", "", 1
        )  # Only replace the first occurrence
    text = text.replace(",", "，").replace("?", "？")
    return text


def format_to_srt(seconds):  # Convert seconds to SRT's timecode
    dt = datetime.datetime(1, 1, 1) + datetime.timedelta(seconds=seconds)
    formatted_time = "{:02d}:{:02d}:{:02d},{:03d}".format(
        dt.hour, dt.minute, dt.second, dt.microsecond // 1000
    )
    return formatted_time

In [5]:
audio_file = "../youtube/guardiola/控制国家的盎撒老爷有多真？在新英格兰起码不假【美国大选地理01】"
mp42wav(audio_file)
chunks = cut_blanks(audio_file, MIN_SILENCE_LEN, SILENCE_THRESH)
audio_file_dir, audio_file_name = audio_paths(audio_file)
for i in range(chunks):
    chunk_audio_file = f"{audio_file_dir}/chunks/{audio_file_name}_chunk{i}"
    transcribe_audio(chunk_audio_file, subtitle_format="txt")
mfa(audio_file_dir)

[0.00s -> 18.20s] 各位好，我是关书迪欧拉。今天来给大家带来一个全新的专题。是的，2026年的中期选举现在正在路上。那么我们现在看到的一个基本情况是这一次的选举被特朗普和反对特朗普的民主党认为是一场在中期的决战。
[18.20s -> 20.44s] 往常其实从美国的历史上来讲
[20.44s -> 23.08s] 并不会出现一个中期选举
[23.08s -> 24.96s] 对于绝大多数的政治
[24.96s -> 27.56s] 乃至于说对于总统个人的政治生命
[27.56s -> 28.38s] 甚至物理生命
[28.38s -> 30.42s] 会有很大影响的这样的一种情况
[30.42s -> 32.82s] 但是我们熟悉时政的
[32.82s -> 34.98s] 尤其美国政治的各位粉丝朋友们
[34.98s -> 35.96s] 那大家也都知道
[35.96s -> 40.00s] 在2025年传宝第二次入主白宫之后
[40.46s -> 42.06s] 短短的一年时间里面
[42.06s -> 44.90s] 是公布了超过300条总统行政令
[44.90s -> 47.18s] 仅次于福安克林罗斯福
[47.18s -> 48.94s] 在历史上已经是
[48.94s -> 50.90s] 可以说虽然没有空前
[50.90s -> 52.10s] 但是大概率要绝后
[52.10s -> 54.86s] 不是骤传宝的这样的一种情况
[54.86s -> 56.04s] 而是在这样的一种
[56.04s -> 57.56s] 大刀阔斧的改革之下
[57.56s -> 60.24s] 应该说2026年的中期选举
[60.24s -> 62.86s] 一旦特朗普失去了国会的支持
[62.86s -> 64.74s] 很有可能就没有办法
[64.74s -> 66.56s] 像现在这样深度的改造
[66.56s -> 68.28s] 美国成为他希望当中的样子
[68.28s -> 69.86s] 乃至于说深度控制美国
[70.32s -> 72.02s] 所以在这样的一种情况下
[72.02s -> 74.20s] 可以说如果中期选举
[74.20s -> 76.78s] 特朗普能够继续掌控两院
[76.78s -> 78.50s] 他的后续的

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107/100  [ 0:00:02 < 0:00:00 , ? it/s ]


 INFO     Found 1 speaker across 107 files, average number of utterances per    
          speaker: 107.0                                                        
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use 1 jobs. Use the --single_speaker flag if you would  
          like to split utterances across jobs regardless of their speaker.     
 INFO     Normalizing text...                                                   


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107/107  [ 0:00:02 < 0:00:00 , ? it/s ]


...

 INFO     Analyzing alignment quality...                                        


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107/107  [ 0:00:03 < 0:00:00 , ? it/s ]
   0% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/107  [ 0:00:00 < -:--:-- , ? it/s ]

 INFO     Exporting alignment TextGrids to ../youtube/guardiola/chunks...       


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107/107  [ 0:00:00 < 0:00:00 , 480 it/s ] ? it/s ]


 INFO     Finished exporting TextGrids to ../youtube/guardiola/chunks!          
 INFO     Done! Everything took 76.729 seconds                                  
